In [ ]:
# @title Setup your Environment
!python -V
!pip install -q openmm MDAnalysis py3Dmol ipywidgets tidynamics

# install compiler
!apt-get -qq update
!apt-get -qq install -y gfortran

# remove existing directory if it exists
import os
import shutil
if os.path.exists("packmol-master"):
    print("Removing existing packmol-master directory...")
    shutil.rmtree("packmol-master")

# download latest Packmol source
!wget -q https://github.com/m3g/packmol/archive/refs/heads/master.zip
!unzip -q master.zip

# compile
%cd packmol-master
!make
%cd ..

# add to PATH

os.environ["PATH"] = "/content/packmol-master:" + os.environ["PATH"]
!which packmol

In [ ]:
#@title Move this Cell up Before Releasing to Students
from openmm.app import *
from openmm import *
from openmm.unit import *
from sys import stdout
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import MDAnalysis as mda
import MDAnalysis.analysis.rdf as rdf
import MDAnalysis.analysis.msd as msd
from scipy.stats import linregress
import py3Dmol
import ipywidgets as widgets
from IPython.display import display

In [ ]:
#@title Define Simulation Parameters


#SIMULATION TEMPERATURE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=298.0,  # default value
    description='Simulation Temperature (K):',
    step=1.0,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simTemperature
    simTemperature = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simTemperature = temp_text.value*kelvin

#BOXSIZE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=48.0,  # default value
    description='Box Size (Angstrom):',
    step=1.0,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global boxsize
    boxsize = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
boxsize = temp_text.value

#TIMESTEP
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=2.0,  # default value
    description='Time step (fs):',
    step=0.5,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simTimestep
    simTimestep = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simTimestep = temp_text.value*femtoseconds

#PRESSURE
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=1.0,  # default value
    description='Simulation pressure (atm):',
    step=0.10,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simPressure
    simPressure = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simPressure = temp_text.value*atmospheres

#STEPS
# create a float text widget to input variable
temp_text = widgets.FloatText(
    value=100000,  # default value
    description='# MD steps:',
    step=100,  # step size
)
# adjust the layout of the widget
temp_text.layout.width = 'auto'
temp_text.style.description_width = 'initial'
# define a function to update the variable
def update_temp_text(change):
    global simNumSteps
    simNumSteps = change.new
# register the update function with the widget
temp_text.observe(update_temp_text, 'value')
# display the widget
display(temp_text)
# access the selected temperature value
simNumSteps = temp_text.value

# Salt
# create an integer text widget to input variable
salt_text = widgets.IntText(
    value=0,  # default value
    description='Number of NaX ion pairs:',
    step=1,  # step size
)
# adjust the layout of the widget
salt_text.layout.width = 'auto'
salt_text.style.description_width = 'initial'
# define a function to update the variable
def update_salt_text(change):
    global salt
    salt = change.new
# register the update function with the widget
salt_text.observe(update_salt_text, 'value')
# display the widget
display(salt_text)
# access the selected salt value
salt = salt_text.value

# Anion identity
anion_selector = widgets.Dropdown(
    options=[('Cl-', 'cl-'), ('F-', 'f-'), ('I-', 'i-')],
    value='cl-',
    description='X anion:',
)
anion_selector.layout.width = 'auto'
anion_selector.style.description_width = 'initial'

def update_anion_selector(change):
    global salt_anion
    salt_anion = change.new

anion_selector.observe(update_anion_selector, 'value')
display(anion_selector)
salt_anion = anion_selector.value

In [ ]:
#@title Store Simulation Parameters
density=0.99705
water=int(boxsize**3*density*1e6*(1e-10)**3/(18.01)*6.02214e23)
concentration_molpercent=salt/water*100
concentration_molar=salt*density*1e3/(water*18.01)
print("Box size of",boxsize,"angstroms at density,",density,"g/cm3 requires ",water," water molecules")
print("Electrolyte concentration (mol%)=",concentration_molpercent)
print("Electrolyte concentration (M)=",concentration_molar)
print(f"Selected NaX salt: Na{salt_anion[:-1].capitalize()}")
print(f"Simulation size: {water} waters")
print(f"Box size: {boxsize} Ang.")
print(f"Simulation temperature: {simTemperature}")
print(f"Simulation pressure: {simPressure}")
print(f"Simulation time step: {simTimestep}")
print(f"# MD steps: {simNumSteps} ; Total simulation length: {simNumSteps * simTimestep} = {simNumSteps*simTimestep / 1000 / 1000 / femtoseconds * nanoseconds}.")
pdbFile = 'system.pdb'

In [ ]:
#@title Generate your Simulation Box
with open("packmol.in", "w") as f:
    f.write("tolerance 2.0\n")
    f.write("filetype pdb\n")
    f.write("output system.pdb\n")
    f.write("\n")

    # Define box and centre
    L = boxsize + 30
    cx = cy = cz = L / 2.0

    # Radii for micelle bias
    r_outer = 0.35 * L     # whole SDS region
    r_head_inner = 0.25 * L
    r_core = 0.15 * L

    # SDS molecules
    f.write("structure sds.pdb\n")
    f.write("  number 60\n")

    # Keep whole molecule in spherical region
    f.write("  inside sphere {0:.2f} {1:.2f} {2:.2f} {3:.2f}\n".format(cx, cy, cz, r_outer))

    # Tail-end carbon (atom 39) pulled toward centre
    f.write("  atoms 39\n")
    f.write("    inside sphere {0:.2f} {1:.2f} {2:.2f} {3:.2f}\n".format(cx, cy, cz, r_core))
    f.write("  end atoms\n")

    # Sulfur (atom 1) pushed outward
    f.write("  atoms 1\n")
    f.write("    outside sphere {0:.2f} {1:.2f} {2:.2f} {3:.2f}\n".format(cx, cy, cz, r_head_inner))
    #f.write("    inside sphere {0:.2f} {1:.2f} {2:.2f} {3:.2f}\n".format(cx, cy, cz, r_outer))
    f.write("  end atoms\n")

    f.write("end structure\n")

    # Sodium counterions (no constraint) from SDS
    f.write("structure na+.pdb\n")
    f.write("  number 60\n")
    f.write("  inside cube 2. 2. 2. {0}\n".format(L))
    f.write("end structure\n")

    # Water
    f.write("structure water.pdb\n")
    f.write("  number {0}\n".format(int(water)))
    f.write("  inside cube 2. 2. 2. {0}\n".format(L))
    f.write("end structure\n")

    # Salt ions (NaX ion pairs)
    if salt > 0:
        f.write("structure na+.pdb\n")
        f.write("  number {0}\n".format(salt))
        f.write("  inside cube 2. 2. 2. {0}\n".format(L))
        f.write("end structure\n")

        f.write(f"structure {salt_anion}.pdb\\n")
        f.write("  number {0}\n".format(salt))
        f.write("  inside cube 2. 2. 2. {0}\n".format(L))
        f.write("end structure\n")

    f.write("\n")
    f.write("pbc {0} {0} {0}\n".format(L))

water_text = '''ATOM      1  O   HOH A   1       4.125  13.679  13.761  1.00  0.00           O
ATOM      2  H1  HOH A   1       4.025  14.428  14.348  1.00  0.00           H
ATOM      3  H2  HOH A   1       4.670  13.062  14.249  1.00  0.00           H'''
na_text = '''ATOM      1  NA  NA  A   1       4.125  13.679  13.761  1.00  0.00          Na'''
cl_text = '''ATOM      1  CL  CL  A   1       4.125  13.679  13.761  1.00  0.00          Cl'''
f_text = '''ATOM      1  F   F   A   1       4.125  13.679  13.761  1.00  0.00           F'''
i_text = '''ATOM      1  I   I   A   1       4.125  13.679  13.761  1.00  0.00           I'''

sds_text = '''REMARK  GENERATED BY CHARMM-GUI (HTTP://WWW.CHARMM-GUI.ORG) VERSION ON APR, 15. 2015.
REMARK  READ PDB, MANIPULATE STRUCTURE IF NEEDED, AND GENERATE TOPOLOGY FILE
REMARK   DATE:    10/17/16     15:50:26      CREATED BY USER: deeperbass
COMPND    UNNAMED
AUTHOR    GENERATED BY OPEN BABEL 2.3.90
ATOM      1  S   SDS S   1       1.453   0.001  -0.001  1.00  0.00           S
ATOM      2  OS1 SDS S   1       1.875   1.510  -0.004  1.00  0.00           O
ATOM      3  OS2 SDS S   1       2.032  -0.509  -1.231  1.00  0.00           O
ATOM      4  OS3 SDS S   1       2.047  -0.566   1.197  1.00  0.00           O
ATOM      5  OS4 SDS S   1       0.001   0.004  -0.001  1.00  0.00           O
ATOM      6  C1  SDS S   1       1.413   2.239   1.128  1.00  0.00           C
ATOM      7  H11 SDS S   1       1.819   1.814   2.077  1.00  0.00           H
ATOM      8  H12 SDS S   1       0.297   2.241   1.180  1.00  0.00           H
ATOM      9  C2  SDS S   1       1.895   3.690   1.000  1.00  0.00           C
ATOM     10  H21 SDS S   1       1.501   4.116   0.051  1.00  0.00           H
ATOM     11  H22 SDS S   1       3.006   3.693   0.938  1.00  0.00           H
ATOM     12  C3  SDS S   1       1.446   4.563   2.179  1.00  0.00           C
ATOM     13  H31 SDS S   1       1.838   4.119   3.121  1.00  0.00           H
ATOM     14  H32 SDS S   1       0.335   4.545   2.234  1.00  0.00           H
ATOM     15  C4  SDS S   1       1.928   6.015   2.059  1.00  0.00           C
ATOM     16  H41 SDS S   1       1.535   6.448   1.112  1.00  0.00           H
ATOM     17  H42 SDS S   1       3.039   6.022   1.999  1.00  0.00           H
ATOM     18  C5  SDS S   1       1.479   6.888   3.239  1.00  0.00           C
ATOM     19  H51 SDS S   1       1.870   6.447   4.184  1.00  0.00           H
ATOM     20  H52 SDS S   1       0.367   6.876   3.296  1.00  0.00           H
ATOM     21  C6  SDS S   1       1.964   8.339   3.121  1.00  0.00           C
ATOM     22  H61 SDS S   1       1.573   8.776   2.175  1.00  0.00           H
ATOM     23  H62 SDS S   1       3.075   8.346   3.062  1.00  0.00           H
ATOM     24  C7  SDS S   1       1.516   9.211   4.302  1.00  0.00           C
ATOM     25  H71 SDS S   1       1.906   8.770   5.247  1.00  0.00           H
ATOM     26  H72 SDS S   1       0.405   9.203   4.358  1.00  0.00           H
ATOM     27  C8  SDS S   1       2.004  10.661   4.185  1.00  0.00           C
ATOM     28  H81 SDS S   1       1.614  11.101   3.240  1.00  0.00           H
ATOM     29  H82 SDS S   1       3.116  10.668   4.128  1.00  0.00           H
ATOM     30  C9  SDS S   1       1.557  11.532   5.367  1.00  0.00           C
ATOM     31  H91 SDS S   1       1.946  11.091   6.312  1.00  0.00           H
ATOM     32  H92 SDS S   1       0.445  11.526   5.423  1.00  0.00           H
ATOM     33  C10 SDS S   1       2.046  12.982   5.251  1.00  0.00           C
ATOM     34 H101 SDS S   1       1.658  13.424   4.307  1.00  0.00           H
ATOM     35 H102 SDS S   1       3.158  12.988   5.195  1.00  0.00           H
ATOM     36  C11 SDS S   1       1.600  13.854   6.433  1.00  0.00           C
ATOM     37 H111 SDS S   1       1.987  13.404   7.376  1.00  0.00           H
ATOM     38 H112 SDS S   1       0.488  13.840   6.488  1.00  0.00           H
ATOM     39  C12 SDS S   1       2.081  15.304   6.332  1.00  0.00           C
ATOM     40 H121 SDS S   1       1.687  15.781   5.410  1.00  0.00           H
ATOM     41 H122 SDS S   1       3.191  15.343   6.300  1.00  0.00           H
ATOM     42 H123 SDS S   1       1.733  15.892   7.209  1.00  0.00           H
CONECT    1    3    2    5    4
CONECT    2    1    6
CONECT    3    1
CONECT    4    1
CONECT    5    1
CONECT    6    2    9    8    7
CONECT    7    6
CONECT    8    6
CONECT    9   10   11    6   12
CONECT   10    9
CONECT   11    9
CONECT   12    9   15   14   13
CONECT   13   12
CONECT   14   12
CONECT   15   16   17   12   18
CONECT   16   15
CONECT   17   15
CONECT   18   15   21   20   19
CONECT   19   18
CONECT   20   18
CONECT   21   22   23   18   24
CONECT   22   21
CONECT   23   21
CONECT   24   21   27   26   25
CONECT   25   24
CONECT   26   24
CONECT   27   28   29   24   30
CONECT   28   27
CONECT   29   27
CONECT   30   27   33   32   31
CONECT   31   30
CONECT   32   30
CONECT   33   34   35   30   36
CONECT   34   33
CONECT   35   33
CONECT   36   33   39   38   37
CONECT   37   36
CONECT   38   36
CONECT   39   40   41   36   42
CONECT   40   39
CONECT   41   39
CONECT   42   39
MASTER        0    0    0    0    0    0    0    0   42    0   42    0
END'''


with open('sds.pdb', 'w') as f:
    f.write(sds_text)
with open('water.pdb', 'w') as f:
    f.write(water_text)
with open('na+.pdb', 'w') as f:
    f.write(na_text)
with open('cl-.pdb', 'w') as f:
    f.write(cl_text)
with open('f-.pdb', 'w') as f:
    f.write(f_text)
with open('i-.pdb', 'w') as f:
    f.write(i_text)

!packmol < packmol.in > packmol.out

with open("packmol.out", "r") as f:
    packmol_output = f.read()

if "Success!" in packmol_output:
    import py3Dmol

    with open("system.pdb", "r") as f:
        pdb_data = f.read()

    view = py3Dmol.view(width=900, height=600)
    # crucial change: keepH=True
    view.addModel(pdb_data, "pdb", {"keepH": True})

    # Default style: water as sticks, ions as spheres
    view.setStyle({}, {})
    view.setStyle({"elem": "O"}, {"stick": {"radius": 0.05}})
    view.addStyle({"elem": "H"}, {"stick": {"radius": 0.05}})
    view.setStyle({"resn": "SDS"}, {"stick": {"radius": 0.15}})
    view.addStyle({"elem": "Na"}, {"sphere": {"scale": 0.7, "color": "blue"}})
    view.addStyle({"elem": "Cl"}, {"sphere": {"scale": 0.7, "color": "green"}})
    view.addStyle({"elem": "F"}, {"sphere": {"scale": 0.7, "color": "pink"}})
    view.addStyle({"elem": "I"}, {"sphere": {"scale": 0.7, "color": "red"}})
    
    view.zoomTo()
    view.show()
else:
    print("Packmol did not finish successfully... Here is the contents of packmol.out:")
    with open('packmol.out', 'r') as f:
        text = f.read()
        print(text)

In [ ]:
#@title Setup & Run your Simulation
pdb = PDBFile('system.pdb')
forcefield = ForceField('charmm36.xml','charmm36/water.xml')
system = forcefield.createSystem(pdb.topology,nonbondedMethod=PME,nonbondedCutoff=14*angstrom,constraints=HBonds,rigidWater=True)
system.addForce(MonteCarloBarostat(simPressure, simTemperature, 1))
integrator = LangevinMiddleIntegrator(simTemperature, 1/picosecond, simTimestep)
simulation = Simulation(pdb.topology, system, integrator)
simulation.context.setPositions(pdb.positions)
#simulation.context.setPeriodicBoxVectors([[3.6, 0, 0], [0, 3.6, 0], [0, 0, 3.6]] * unit.nanometers)
print("Running Energy Minimisation:")
t0 = time.time()
simulation.minimizeEnergy(maxIterations=0)
t1 = time.time()
minTime = t1-t0
print("Minimisation took {} seconds".format(minTime))
simulation.reporters.append(DCDReporter('sds.dcd', 10))
simulation.reporters.append(StateDataReporter(
    'sds_data.csv',
    10,
    step=True,
    time=True,
    temperature=True,
    potentialEnergy=True,
    kineticEnergy=True,
    totalEnergy=True,
    volume=True,
    density=True
))
state = simulation.context.getState(getPositions=False, getVelocities=False, getForces=False, getEnergy=False, getParameters=False)
box_vectors = state.getPeriodicBoxVectors()
# Run the simulation:
print('Start Simulation')
t0 = time.time()
simulation.step(simNumSteps)
t1 = time.time()
simTime = t1-t0
print("Simulation took {} seconds for {} timesteps".format(simTime,simNumSteps))

In [ ]:
#@title Visualise your Trajectory

u = mda.Universe("system.pdb", "sds.dcd")

stride = 10

# Keep only SDS + Na+, omit water
sel = u.select_atoms("resname SDS or name NA or name Na or element Na or name Cl or element Cl or name F or element F or name I or element I")

# Build XYZ trajectory in memory
xyz_frames = []

for ts in u.trajectory[::stride]:
    lines = [f"{len(sel)}", f"Frame {ts.frame}"]

    for atom in sel:
        x, y, z = atom.position
        name = atom.name.upper()

        if name.startswith("H"):
            elem = "H"
        elif name.startswith("O"):
            elem = "O"
        elif name.startswith("NA"):
            elem = "Na"
        elif name.startswith("CL"):
            elem = "Cl"
        elif name.startswith("F"):
            elem = "F"
        elif name.startswith("I"):
            elem = "I"
        elif name.startswith("S"):
            elem = "S"
        elif name.startswith("C"):
            elem = "C"
        else:
            elem = atom.name[0]

        lines.append(f"{elem} {x:.3f} {y:.3f} {z:.3f}")

    xyz_frames.append("\n".join(lines))

# Combine all frames into one string
xyz_data = "\n".join(xyz_frames)

view = py3Dmol.view(width=900, height=550)
view.addModelsAsFrames(xyz_data, "xyz")

# Global style for SDS
view.setStyle({}, {"stick": {"radius": 0.12}})

# Make sulfur stand out a bit
view.addStyle({"elem": "S"}, {"sphere": {"scale": 0.35}})

# Highlight Na+
view.addStyle({"elem": "Na"}, {"sphere": {"scale": 0.7, "color": "blue"}})
view.addStyle({"elem": "Cl"}, {"sphere": {"scale": 0.7, "color": "green"}})
view.addStyle({"elem": "F"}, {"sphere": {"scale": 0.7, "color": "pink"}})
view.addStyle({"elem": "I"}, {"sphere": {"scale": 0.7, "color": "red"}})

view.zoomTo()
view.animate({"interval": 100})
view.show()


In [ ]:
#@title Check for Equilibration...
df = pd.read_csv('sds_data.csv')

time_col = 'Time (ps)'
if 'Time (ps)' not in df.columns:
    if 'Time' in df.columns:
        time_col = 'Time'
    else:
        print("Warning: 'Time (ps)' column not found. Please check CSV header.")
        time_col = 'Step'

property_columns = {
    'Potential Energy': ['Potential Energy (kJ/mole)', 'Potential Energy'],
    'Kinetic Energy': ['Kinetic Energy (kJ/mole)', 'Kinetic Energy'],
    'Total Energy': ['Total Energy (kJ/mole)', 'Total Energy'],
    'Temperature': ['Temperature (K)', 'Temperature'],
    'Volume': ['Box Volume (nm^3)', 'Volume (nm^3)', 'Volume'],
    'Density': ['Density (g/mL)', 'Density']
}

def get_column_name(property_name):
    for column_name in property_columns[property_name]:
        if column_name in df.columns:
            return column_name
    raise KeyError(f"No matching column found for {property_name}.")

plot_selector = widgets.Dropdown(
    options=list(property_columns.keys()),
    value='Potential Energy',
    description='Property:',
)
plot_selector.layout.width = 'auto'
plot_selector.style.description_width = 'initial'

start_time = float(df[time_col].min())
end_time = float(df[time_col].max())

start_time_widget = widgets.FloatText(
    value=start_time,
    description='Start time:',
    step=1.0,
)
start_time_widget.layout.width = 'auto'
start_time_widget.style.description_width = 'initial'

end_time_widget = widgets.FloatText(
    value=end_time,
    description='End time:',
    step=1.0,
)
end_time_widget.layout.width = 'auto'
end_time_widget.style.description_width = 'initial'

output = widgets.Output()

def plot_property(change=None):
    selected_property = plot_selector.value
    start = start_time_widget.value
    end = end_time_widget.value
    with output:
        output.clear_output(wait=True)
        try:
            column_name = get_column_name(selected_property)
        except KeyError as err:
            print(err)
            return

        if start > end:
            print("Start time must be less than or equal to end time.")
            return

        mask = (df[time_col] >= start) & (df[time_col] <= end)
        filtered_df = df.loc[mask]

        if filtered_df.empty:
            print("No data found in the selected time interval.")
            return

        mean_value = filtered_df[column_name].mean()
        std_value = filtered_df[column_name].std()
        print(f'Mean {selected_property}: {mean_value:.5f}')
        print(f'Standard deviation {selected_property}: {std_value:.5f}')
        print()

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.plot(filtered_df[time_col], filtered_df[column_name])
        ax.set_xlabel(time_col)
        ax.set_ylabel(column_name)
        ax.set_title(f'{selected_property} vs {time_col} ({start} to {end})')
        ax.grid(True)
        plt.tight_layout()
        plt.show()

plot_selector.observe(plot_property, names='value')
start_time_widget.observe(plot_property, names='value')
end_time_widget.observe(plot_property, names='value')
display(plot_selector)
display(start_time_widget)
display(end_time_widget)
display(output)
plot_property()

In [ ]:
#@title Analyse your Micelle!

u = mda.Universe("system.pdb", "sds.dcd")
micelle = u.select_atoms("resname SDS")
na_cations = u.select_atoms("name NA")[:60]
sds_sulfur = u.select_atoms("resname SDS and name S")
sds_tail_carbons = u.select_atoms("resname SDS and name C1 C2 C3 C4 C5 C6 C7 C8 C9 C10 C11 C12")
sds_c1_carbons = u.select_atoms("resname SDS and name C1")
sds_c12_carbons = u.select_atoms("resname SDS and name C12")
com = micelle.center_of_mass()
water = u.select_atoms("resname HOH")

analysis_selector = widgets.Dropdown(
    options=[
        "Micellar Radius",
        "Micelle Eccentricity",
        "Na-Sulfate RDF",
        "Counterion Diffusion",
        "Distances to Micelle COM",
        "Chain-COM Angles"
    ],
    value="Micellar Radius",
    description="Analysis:",
)
analysis_selector.layout.width = 'auto'
analysis_selector.style.description_width = 'initial'

frame_times = np.array([ts.time for ts in u.trajectory])
start_time_widget = widgets.FloatText(
    value=float(frame_times.min()),
    description='Start time (ps):',
    step=1.0,
)
start_time_widget.layout.width = 'auto'
start_time_widget.style.description_width = 'initial'

end_time_widget = widgets.FloatText(
    value=float(frame_times.max()),
    description='End time (ps):',
    step=1.0,
)
end_time_widget.layout.width = 'auto'
end_time_widget.style.description_width = 'initial'

export_button = widgets.Button(
    description='Write plotted data to CSV',
    button_style='success',
    disabled=True,
)
export_button.layout.width = 'auto'

output = widgets.Output()
export_output = widgets.Output()
current_export_df = None
current_export_filename = None

def slugify(text):
    return re.sub(r'[^0-9A-Za-z]+', '_', str(text)).strip('_')

def build_export_filename(analysis_name, start_time, end_time):
    anion_label = salt_anion[:-1].capitalize()
    parts = [
        slugify(analysis_name),
        f"T{simTemperature.value_in_unit(kelvin):g}K",
        f"P{simPressure.value_in_unit(atmospheres):g}atm",
        f"box{boxsize:g}A",
        f"dt{simTimestep.value_in_unit(femtoseconds):g}fs",
        f"steps{int(simNumSteps)}",
        f"NaXpairs{int(salt)}",
        f"anion{anion_label}",
        f"start{start_time:g}ps",
        f"end{end_time:g}ps",
    ]
    return '_'.join(parts) + '.csv'

def set_export_data(df, analysis_name, start_time, end_time):
    global current_export_df, current_export_filename
    current_export_df = df
    current_export_filename = build_export_filename(analysis_name, start_time, end_time)
    export_button.disabled = df is None or df.empty

def write_current_data_to_csv(button):
    with export_output:
        export_output.clear_output(wait=True)
        if current_export_df is None or current_export_filename is None or current_export_df.empty:
            print('No plotted data available to export.')
            return
        current_export_df.to_csv(current_export_filename, index=False)
        print(f"Wrote {len(current_export_df)} rows to {current_export_filename}")

export_button.on_click(write_current_data_to_csv)

def eccentricity_from_atomgroup(ag):
    coords = ag.positions.copy()
    masses = ag.masses
    com = np.average(coords, axis=0, weights=masses)
    r = coords - com

    x, y, z = r[:, 0], r[:, 1], r[:, 2]
    m = masses

    inertia_tensor = np.array([
        [np.sum(m * (y * y + z * z)), -np.sum(m * x * y), -np.sum(m * x * z)],
        [-np.sum(m * x * y), np.sum(m * (x * x + z * z)), -np.sum(m * y * z)],
        [-np.sum(m * x * z), -np.sum(m * y * z), np.sum(m * (x * x + y * y))]
    ])

    evals = np.linalg.eigvalsh(inertia_tensor)
    return 1.0 - evals[0] / np.mean(evals), evals

def get_frame_window(start_time, end_time):
    if start_time > end_time:
        raise ValueError('Start time must be less than or equal to end time.')

    mask = (frame_times >= start_time) & (frame_times <= end_time)
    indices = np.where(mask)[0]
    if len(indices) == 0:
        raise ValueError('No trajectory frames found in the selected time range.')

    return indices[0], indices[-1] + 1, frame_times[indices]

def angles_from_points(a_points, b_points, c_point):
    ba = a_points - b_points
    bc = c_point - b_points
    ba_norm = np.linalg.norm(ba, axis=1)
    bc_norm = np.linalg.norm(bc, axis=1)
    valid = (ba_norm > 0) & (bc_norm > 0)
    cos_theta = np.sum(ba[valid] * bc[valid], axis=1) / (ba_norm[valid] * bc_norm[valid])
    cos_theta = np.clip(cos_theta, -1.0, 1.0)
    return np.degrees(np.arccos(cos_theta))

def get_na_diffusion_subsets(frame_index):
    u.trajectory[frame_index]
    com = micelle.center_of_mass()
    distances = np.linalg.norm(na_cations.positions - com, axis=1)
    subset_specs = [
        {
            'label': 'Na+ 16-20 A',
            'export_key': 'na_16_20',
            'color': 'tab:blue',
            'mask': (distances >= 16.0) & (distances < 20.0),
        },
        {
            'label': 'Na+ 20-22.5 A',
            'export_key': 'na_20_22p5',
            'color': 'tab:orange',
            'mask': (distances >= 20.0) & (distances < 22.5),
        },
        {
            'label': 'Na+ 22.5-26 A',
            'export_key': 'na_22p5_26',
            'color': 'tab:green',
            'mask': (distances >= 22.5) & (distances <= 26.0),
        },
        {
            'label': 'Na+ >26 A',
            'export_key': 'na_gt_26',
            'color': 'tab:red',
            'mask': distances > 26.0,
        },
    ]

    subsets = []
    for spec in subset_specs:
        indices = np.where(spec['mask'])[0]
        subsets.append({
            'label': spec['label'],
            'export_key': spec['export_key'],
            'color': spec['color'],
            'indices': indices,
            'atomgroup': na_cations[indices],
        })
    return subsets

def plot_selected_analysis(change=None):
    selected_analysis = analysis_selector.value
    start_time = start_time_widget.value
    end_time = end_time_widget.value
    full_start_time = float(frame_times.min())
    full_end_time = float(frame_times.max())
    with output:
        output.clear_output(wait=True)
        export_output.clear_output(wait=True)
        set_export_data(None, selected_analysis, start_time, end_time)
        try:
            start_frame, stop_frame, selected_times = get_frame_window(start_time, end_time)
        except ValueError as err:
            print(err)
            return

        if start_time > full_start_time or end_time < full_end_time:
            print("Warning: you are analysing only part of the MD trajectory, not the full trajectory.")
            print()

        if selected_analysis == "Micellar Radius":
            rg = []
            time = []
            for ts in u.trajectory[start_frame:stop_frame]:
                rg.append(micelle.radius_of_gyration())
                time.append(ts.time)

            rg = np.array(rg) * sqrt(5/3)
            print(f"Mean radius (+/- std dev): {rg.mean():.4f} +/- {rg.std():.4f} Å")
            print()
            set_export_data(pd.DataFrame({
                'time_ps': time,
                'micellar_radius_A': rg
            }), selected_analysis, start_time, end_time)
            plt.figure(figsize=(8, 5))
            plt.plot(time, rg)
            plt.xlabel("Time (ps)")
            plt.ylabel("Radius (Å)")
            #plt.title(f"Micellar Radius vs time ({start_time} to {end_time} ps)")
            plt.grid(False)
            plt.show()

        elif selected_analysis == "Micelle Eccentricity":
            times = []
            ecc = []
            for ts in u.trajectory[start_frame:stop_frame]:
                e, _ = eccentricity_from_atomgroup(micelle)
                times.append(ts.time)
                ecc.append(e)

            ecc = np.array(ecc)
            print(f"Mean eccentricity (+/- std dev): {ecc.mean():.4f} +/- {ecc.std():.4f}")
            print()
            set_export_data(pd.DataFrame({
                'time_ps': times,
                'eccentricity': ecc
            }), selected_analysis, start_time, end_time)
            plt.figure(figsize=(8, 5))
            plt.plot(times, ecc)
            plt.xlabel("Time (ps)")
            plt.ylabel("Eccentricity")
            #plt.title(f"Micelle eccentricity vs time ({start_time} to {end_time} ps)")
            plt.grid(False)
            plt.show()

        elif selected_analysis == "Na-Sulfate RDF":
            R = rdf.InterRDF(na_cations, sds_sulfur, exclusion_block=(1, 1), nbins=75, range=(0.0, 15.0))
            R.run(start=start_frame, stop=stop_frame)
            set_export_data(pd.DataFrame({
                'distance_A': R.results.bins,
                'g_r': R.results.rdf
            }), selected_analysis, start_time, end_time)
            plt.figure(figsize=(8, 5))
            plt.plot(R.results.bins, R.results.rdf)
            plt.xlabel("Distance (Å)")
            plt.ylabel("g(r)")
            #plt.title(f"RDF between Na+ counterions and SDS sulfur atoms ({start_time} to {end_time} ps)")
            plt.grid(False)
            plt.show()

        elif selected_analysis == "Counterion Diffusion":
            if stop_frame - start_frame < 2:
                print("Not enough frames in trajectory to calculate MSD.")
                return

            subsets = get_na_diffusion_subsets(start_frame)
            timestep = simTimestep * stride
            export_dict = {}
            lagtimes = None
            plotted_any_subset = False
            print('Na+ subsets are defined from distances to the micelle COM in the first frame of the selected time window.')
            print('Note: the 22.5-26 A and >25 A ranges overlap between 25 and 26 A.')
            print()

            fig, ax = plt.subplots(figsize=(8, 5))

            for subset in subsets:
                n_atoms = len(subset['atomgroup'])
                print(f"{subset['label']}: {n_atoms} Na+ ions")
                if n_atoms == 0:
                    print('  No ions in this subset; skipping MSD calculation.')
                    print()
                    continue

                MSD = msd.EinsteinMSD(subset['atomgroup'], msd_type='xyz', fft=True)
                MSD.run(start=start_frame, stop=stop_frame)
                msd_values = MSD.results.timeseries
                nframes = MSD.n_frames
                subset_lagtimes = np.arange(nframes) * timestep.value_in_unit(picosecond)
                fit_values = np.full(len(subset_lagtimes), np.nan)

                start_index = min(2500, max(len(subset_lagtimes) - 2, 0))
                end_index = min(7500, len(subset_lagtimes))
                if end_index - start_index < 2:
                    start_index = 0
                    end_index = len(subset_lagtimes)

                ax.plot(subset_lagtimes, msd_values, color=subset['color'], label=subset['label'])
                plotted_any_subset = True
                lagtimes = subset_lagtimes
                export_dict[f"{subset['export_key']}_msd_A2"] = msd_values
                export_dict[f"{subset['export_key']}_linear_fit_A2"] = fit_values

                if end_index - start_index >= 2:
                    linear_model = linregress(subset_lagtimes[start_index:end_index], msd_values[start_index:end_index])
                    slope = linear_model.slope
                    intercept = linear_model.intercept
                    error = linear_model.stderr
                    fit_x = subset_lagtimes[start_index:end_index]
                    fit_y = slope * fit_x + intercept
                    fit_values[start_index:end_index] = fit_y
                    export_dict[f"{subset['export_key']}_linear_fit_A2"] = fit_values
                    ax.plot(fit_x, fit_y, linestyle='--', color=subset['color'])
                    D = slope / 6 * (1e-20) / (1e-12) * meter * meter / second
                    pm_D = error / 6 * (1e-20) / (1e-12) * meter * meter / second
                    print('  Self diffusion coefficient is', D, '+/-', pm_D)
                    print(f"  Computed from frames between {selected_times[0]} and {selected_times[-1]} ps")
                else:
                    print('  Not enough lag-time points available for a linear fit.')
                print()

            if not plotted_any_subset:
                plt.close(fig)
                print('No Na+ subsets contained any ions in the selected time window.')
                return

            ax.set_xlabel('Time (ps)')
            ax.set_ylabel('MSD (Å$^2$)')
            #ax.set_title(f'Mean squared displacement of Na+ cations ({start_time} to {end_time} ps)')
            ax.legend()
            ax.grid(False)
            export_dict = {'lagtime_ps': lagtimes, **export_dict}
            set_export_data(pd.DataFrame(export_dict), selected_analysis, start_time, end_time)
            plt.show()

        elif selected_analysis == "Distances to Micelle COM":
            na_distances = []
            sulfur_distances = []
            tail_distances = []
            water_distances = []

            for ts in u.trajectory[start_frame:stop_frame]:
                com = micelle.center_of_mass()
                na_distances.append(np.linalg.norm(na_cations.positions - com, axis=1))
                sulfur_distances.append(np.linalg.norm(sds_sulfur.positions - com, axis=1))
                tail_distances.append(np.linalg.norm(sds_tail_carbons.positions - com, axis=1))
                water_distances.append(np.linalg.norm(water.positions - com, axis=1))

            na_distances = np.array(na_distances).ravel()
            sulfur_distances = np.array(sulfur_distances).ravel()
            tail_distances = np.array(tail_distances).ravel()
            water_distances = np.array(water_distances).ravel()

            print("Distances are computed between each individual atom and the micelle centre of mass (COM).")
            print(f"Na+ distance samples analysed: {len(na_distances)}")
            print(f"SDS sulfur distance samples analysed: {len(sulfur_distances)}")
            print(f"SDS tail carbon distance samples analysed: {len(tail_distances)}")
            print(f"Water distance samples analysed: {len(water_distances)}")
            print()
            set_export_data(pd.DataFrame({
                'na_to_com_A': pd.Series(na_distances),
                'sulfur_to_com_A': pd.Series(sulfur_distances),
                'tail_carbon_to_com_A': pd.Series(tail_distances),
                'water_to_com_A': pd.Series(water_distances)
            }), selected_analysis, start_time, end_time)

            plt.figure(figsize=(10, 6))
            plt.hist(na_distances, bins=200, density=True, histtype='step', linewidth=2.0, color='tab:blue', label='Na+ to COM')
            plt.hist(sulfur_distances, bins=200, density=True, histtype='step', linewidth=2.0, color='tab:orange', label='S to COM')
            plt.hist(tail_distances, bins=200, density=True, histtype='step', linewidth=2.0, color='tab:green', label='Tail C to COM')
            plt.hist(water_distances, bins=200, density=True, histtype='step', linewidth=2.0, color='tab:purple', label='Water to COM')
            plt.xlabel("Distance to Micelle c.o.m. (Å)")
            plt.ylabel("Probability")
            #plt.title(f"Distance distributions to micelle COM ({start_time} to {end_time} ps)")
            plt.legend()
            plt.grid(False)
            plt.show()

        elif selected_analysis == "Chain-COM Angles":
            if len(sds_c1_carbons) != len(sds_c12_carbons):
                print("C1 and C12 selections must contain the same number of atoms.")
                return

            angles = []
            for ts in u.trajectory[start_frame:stop_frame]:
                com = micelle.center_of_mass()
                frame_angles = angles_from_points(sds_c12_carbons.positions, sds_c1_carbons.positions, com)
                angles.append(frame_angles)

            angles = np.array(angles).ravel()
            print("Angles are computed as C12-C1-COM for each SDS molecule.")
            print(f"Angle samples analysed: {len(angles)}")
            print(f"Mean angle (+/- std dev): {angles.mean():.4f} +/- {angles.std():.4f} degrees")
            print()
            set_export_data(pd.DataFrame({
                'chain_com_angle_deg': angles
            }), selected_analysis, start_time, end_time)

            plt.figure(figsize=(10, 6))
            plt.hist(angles, bins=200, density=True, histtype='step', linewidth=2.0, color='tab:red', label='C12-C1-COM')
            plt.xlabel("Angle (degrees)")
            plt.ylabel("Probability")
            plt.legend()
            plt.grid(False)
            plt.show()

analysis_selector.observe(plot_selected_analysis, names='value')
start_time_widget.observe(plot_selected_analysis, names='value')
end_time_widget.observe(plot_selected_analysis, names='value')
display(analysis_selector)
display(start_time_widget)
display(end_time_widget)
display(export_button)
display(export_output)
display(output)
plot_selected_analysis()

/opt/homebrew/Caskroom/miniconda/base/envs/openmm/lib/python3.12/site-packages/MDAnalysis/coordinates/DCD.py:171: DeprecationWarning: DCDReader currently makes independent timesteps by copying self.ts while other readers update self.ts inplace. This behavior will be changed in 3.0 to be the same as other readers. Read more at https://github.com/MDAnalysis/mdanalysis/issues/3889 to learn if this change in behavior might affect you.
  warnings.warn("DCDReader currently makes independent timesteps"


Dropdown(description='Analysis:', layout=Layout(width='auto'), options=('Micellar Radius', 'Micelle Eccentrici…

FloatText(value=0.020000000059628115, description='Start time (ps):', layout=Layout(width='auto'), step=1.0, s…

FloatText(value=200.00000059628115, description='End time (ps):', layout=Layout(width='auto'), step=1.0, style…

Button(button_style='success', description='Write plotted data to CSV', disabled=True, layout=Layout(width='au…

Output()

Output()